In [1]:
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import os
import math
import seaborn as sns
import time


from kan_convolutional.KANLinear import KANLinear
from kan_convolutional.KANConv import KAN_Convolutional_Layer
from kan_convolutional import convolution 

In [2]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Malimg(Dataset):
    def __init__(self, root_dirs, transform=None):
        self.transform = transform
        self.image_files = []
        self.labels = []
        self.class_names = []

        for root_dir in root_dirs:
            for label, subfolder in enumerate(os.listdir(root_dir)):
                subfolder_path = os.path.join(root_dir, subfolder)
                if os.path.isdir(subfolder_path):
                    if subfolder not in self.class_names:
                        self.class_names.append(subfolder)
                    label = self.class_names.index(subfolder)
                    for img_file in os.listdir(subfolder_path):
                        if img_file.endswith('.png'):
                            self.image_files.append(os.path.join(subfolder_path, img_file))
                            self.labels.append(label)

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        image = Image.open(img_name).convert('L')
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]
        return image, label

    def get_class_names(self):
        return self.class_names

root_dirs = [
    "C:\\Users\\Rajat S Chakraborty\\Desktop\\workingr8now\\256\\celeb\\aug"
]

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = Malimg(root_dirs=root_dirs, transform=transform)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ConvolutionalKAN(nn.Module):
    def __init__(self):
        super(ConvolutionalKAN, self).__init__()
        
        self.conv1 = nn.Conv2d(1, 8, kernel_size=2, padding=1)
        
        self.conv2 = nn.Conv2d(8, 16, kernel_size=2, padding=1)

        self.maxpool = nn.MaxPool2d(kernel_size=(2, 2))

        self.flatten = nn.Flatten()

        self.kan1 = KANLinear(
            in_features=16 * 32 * 32,
            out_features=2,
            grid_size=10,
            spline_order=3,
            scale_noise=0.01,
            scale_base=1,
            scale_spline=1,
            base_activation=nn.SiLU,
            grid_eps=0.02,
            grid_range=[0, 1]
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.maxpool(x)
        
        x = self.conv2(x)
        x = self.maxpool(x)
        
        x = self.flatten(x)
        x = self.kan1(x)
        
        x = F.log_softmax(x, dim=1)  

        return x
    
model = ConvolutionalKAN()
model = model.to(device)

print(next(model.parameters()).device)

cuda:0


In [4]:
import time
import torch
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ConvolutionalKAN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

start_time = time.time()

for epoch in range(10): 
    epoch_start_time = time.time()
    model.train()
    running_loss = 0.0

    with tqdm(train_loader, unit="batch") as tepoch:
        for images, labels in tepoch:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            tepoch.set_description(f"Epoch [{epoch+1}/10]")
            tepoch.set_postfix(loss=running_loss / len(tepoch))

    epoch_time = time.time() - epoch_start_time
    print(f'Epoch [{epoch + 1}/10], Loss: {running_loss / len(train_loader):.4f}, Time elapsed: {epoch_time:.2f} seconds')

total_time = time.time() - start_time
print(f"Training completed in: {total_time:.2f} seconds")

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())  
        all_labels.extend(labels.cpu().numpy())  

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='macro')
recall = recall_score(all_labels, all_preds, average='macro')
f1 = f1_score(all_labels, all_preds, average='macro')

print(f'Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}')

torch.save(model.state_dict(), 'kan_c.pth')

Epoch [1/10]: 100%|██████████| 9232/9232 [51:38<00:00,  2.98batch/s, loss=0.438] 


Epoch [1/10], Loss: 0.4385, Time elapsed: 3098.04 seconds


Epoch [2/10]: 100%|██████████| 9232/9232 [39:19<00:00,  3.91batch/s, loss=0.371] 


Epoch [2/10], Loss: 0.3707, Time elapsed: 2359.45 seconds


Epoch [3/10]: 100%|██████████| 9232/9232 [37:29<00:00,  4.10batch/s, loss=0.343] 


...

Epoch [8/10], Loss: 0.2899, Time elapsed: 2159.34 seconds


Epoch [9/10]: 100%|██████████| 9232/9232 [36:24<00:00,  4.23batch/s, loss=0.281] 


Epoch [9/10], Loss: 0.2807, Time elapsed: 2184.24 seconds


Epoch [10/10]: 100%|██████████| 9232/9232 [35:57<00:00,  4.28batch/s, loss=0.277] 


Epoch [10/10], Loss: 0.2771, Time elapsed: 2157.16 seconds
Training completed in: 22916.25 seconds
Accuracy: 0.8534, Precision: 0.8621, Recall: 0.8507, F1 Score: 0.8517
